In [ ]:
import os, json
import torch
import transformers
import pandas as pd
import numpy as np

import transformers 
from transformers import AutoTokenizer

import MeMoHF
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import (
    seed_everything,
    load_model_and_tokenizer,
    save_data_to_disk,
    load_from_disk
)

import datasets 
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

os.environ['CUDA_VISIBLE_DEVICES'] = '0'



In [ ]:
from learning_evaluation import create_large_sample

In [ ]:
data_path = '/home/davide/.cache/huggingface/datasets/wikipedia/20200501.en/1.0.0/009f923d9b6dd00c00c8cdc7f408f2b47f45dd4f5fb7982a21f9448f4afbe475/wikipedia-train.arrow'
data = dict(
    train=Dataset.from_file(data_path)
)

In [ ]:
data['train'].select_columns('text')

In [ ]:
import learning_evaluation
from learning_evaluation import create_all_datasets

# create_all_datasets(main_data_dir='training_data')


In [ ]:
def convert_text_into_cfg(text):
    cfg_list = [
        (param.split('=['))
        for param in text.split(']-')
    ]
    return {
        param[0]:param[1]
        for param in cfg_list
    }

convert_text_into_cfg(os.path.basename('bla/di/bla/max_length=[1024]-d=[1024]-l=[4]-h=[4]-batch_size=[64]-save_every_k_batches=[3]-data_name=[n=1000]-seed=[42]-batch_id=[3]'))

In [ ]:
import pandas as pd

# Sample DataFrame
df = pd.DataFrame([
    {'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'},
    {'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'},
    {'lr': 0.01, 'batch_size': 64, 'optimizer': 'adam'},
])

# Dictionary with a subset of keys
partial_experiment = {'lr': 0.01, 'optimizer': 'adam'}

# Subset DataFrame to the relevant columns and compare
match = (df[partial_experiment.keys()] == pd.Series(partial_experiment)).all(axis=1)

# Check if any row matches the subset
exists = match.any()

print("Subset match exists:", exists)

In [ ]:
import pandas as pd

# Existing DataFrame
df = pd.DataFrame([
    {'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'},
    {'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'},
])

# New row as a dictionary
new_row = {'lr': 0.005, 'batch_size': 128, 'optimizer': 'adam'}

# Add the new row
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

print(df)

Experiments with sequence view augmentation for computing multiple sequences in parallel given [batch_size] texts

In [ ]:
import torch 
import torch.nn.functional as F

# Example input
batch_size = 3
seq_len = 4
hidden_dim = 2
window_size = 2

# x = torch.randn(batch_size, seq_len, hidden_dim)  # (B, S, H)
x = torch.tensor(
    [
        [[1,2],[3,4],[5,6],[7,8]],
        [[9,10],[11,12],[13,14],[15,16]],
        [[17,18],[19,20],[21,22],[23,24]],
    ],
    dtype=torch.int32
)
y = torch.tensor(
    [
        [1,2,3,4],
        [5,6,7,8],
        [9,10,11,12],
    ],
    dtype=torch.int32
)
print(x)
print(y)

In [ ]:
num_windows = seq_len - window_size + 1 
x_windows = x.permute(0, 2, 1) # (B, H, S)
# Now unfold sequence dimension (dim=2)
x_windows = x_windows.unfold(dimension=2, size=window_size, step=1) # (B, H, num_windows, window_size)
# Now bring it back to (B, num_windows, window_size, H)
x_windows = x_windows.permute(0, 2, 3, 1) # (B, num_windows, window_size, H)
# Reshape to (B * num_windows, window_size, H)
x_windows = x_windows.contiguous().view(-1, window_size, hidden_dim)
print(x_windows)

y_windows = y[:, 1:].unfold(dimension=1, size=1, step=1)
y_windows = y_windows.contiguous().view(-1, 1)
print(y_windows)

In [ ]:
new_hidden_dim = hidden_dim
final_x = x_windows.sum(dim=1)
final_x = final_x.view(batch_size, num_windows, new_hidden_dim)
final_y = y_windows.view(batch_size, -1)

print(final_x)
print(final_y)

MEMO Debugging

In [1]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

# d, h, l = 1024, 4, 2
# chunk_length = 16
d,h,l = 2048, 4, 1
chunk_length = 12 

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
               hidden_size=d, 
               num_hidden_layers=l,
               num_attention_heads=h,
               chunk_length=chunk_length,
               bos_token_id=tokenizer.bos_token_id,
               eos_token_id=tokenizer.eos_token_id,
               pad_token_id=tokenizer.pad_token_id,
              )

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


/opt/dev/anaconda3/envs/deepai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/dev/anaconda3/envs/deepai/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.


Setting pad token and pad token id = <|endoftext|>, 0
MeMo embedding initilialization
MeMo embedding initilialization
MeMo embedding initilialization
GPU: NVIDIA GeForce RTX 2060 is available.


In [2]:
with open("../testo_di_prova.txt") as my_first_text_f:
    my_first_text = ''.join(my_first_text_f.read().split()[:50])
with open("../testo_di_prova2.txt") as my_first_text_f:
    my_second_text = my_first_text_f.read()

In [3]:
# first_text = ["a b c d e f g h i j k l m n o p q"]
# first_text = ["a b c d e f g h i j k l m n o"]
# memo_input = tokenizer.get_text_batch_encoding(first_text * 2)

# print(tokenizer.decode(memo_input['input_ids'][0]))
# print(tokenizer.decode(memo_input['labels'][0]))

# print(memo_input['input_ids'])
# print(memo_input['labels'])

memo_input_1 = tokenizer.get_text_batch_encoding([my_first_text]*1)  # Writing the same doc 8 times to stress the memorization with batch
memo_input_2 = tokenizer.get_text_batch_encoding([my_second_text]*1) # Writing the same doc 8 times to stress the memorization with batch

print(tokenizer.decode(memo_input_1['input_ids'][0]))
print(tokenizer.decode(memo_input_1['labels'][0]))

print(memo_input_1['input_ids'])
print(memo_input_1['labels'])

Token indices sequence length is longer than the specified maximum sequence length for this model (98 > 13). Running this sequence through the model will result in indexing errors


<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>CosimodiGiovann
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>CosimodiGiovannide
tensor([[    0,     0,     0,     0,     0,     0, 38577,   303, 26206, 30206,
           729,  1136],
        [    8, 10818,   280,   301, 42136,   300, 29113, 41380,    80,    49,
           727, 44001],
        [    9, 15723,   445,  2721,    13,  1630,  1178,  5142,  3381,  1012,
          2511,  1253],
        [ 1266,    74,    13,    18, 11703,   356, 39337,  1047,  1540,    10,
          3539,  8766],
        [29486,  4173,  2275,   266,  2942,   250,  1562, 10356,    13, 13374,
           375, 30029],
        [  514, 26206, 15723,   445,  2721,   554,  3428,   276,   297, 26206,
          9710,   351],
        [  300, 28017,   351,   437,  2320,   312,   304, 19702, 10818, 12706,
            15,    49],
        [  251,   580,   423,   297,  1094,   280,   412, 16406, 46775,   328,
           317,   274]

In [ ]:
# evaluation method
from MeMoHF.evaluating_memo import EvaluationUpdateNew
evaluation = EvaluationUpdateNew()

def perform_evaluation(model, tokenizer, text, starting_point=1):
    model.eval()
    batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=text)
    with torch.no_grad():
        outs, pretokenized_score = model.forward_with_loss( #model.forward_with_loss_parallelized_efficient( #model.forward_with_loss_parallelized_efficient( #model.forward_with_loss_parallelized(
            batch_inputs=batch_inputs,
            compute_accuracy=True,
            return_dict=True
        )

    # memo_input = tokenizer.get_text_batch_encoding(text=text)
    # with torch.no_grad():
    #     pretokenized_score = evaluation.check_pretokenized(
    #         model=model,
    #         tokenizer=tokenizer,
    #         input_ids=memo_input['input_ids'],
    #         starting_point=starting_point
    #     )
    loss = outs['loss']
    del outs
    model.train()
    torch.cuda.empty_cache()
    return dict(
        out_loss=loss,
        token_accuracy=pretokenized_score
    )

In [5]:
# results = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=first_text*8
# )
# print(results)

# results_2 = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=first_text,
#     starting_point=None
# )
# print(results_2)

results_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)


100%|██████████| 98/98 [00:00<00:00, 177.75it/s]


{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.0, 'correct_tokens': 0, 'tot_tokens': 90}}


100%|██████████| 1264/1264 [00:12<00:00, 100.15it/s]

{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.0, 'correct_tokens': 0, 'tot_tokens': 1166}}


In [6]:
# memorize input
# model.memorize_text(memo_input)
model.memorize_text(memo_input_1)

In [7]:
# post-edit evaluation
# results = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=first_text*8
# )
# print(results)

# results_2 = perform_evaluation(
#     model=model,
#     tokenizer=tokenizer,
#     text=first_text,
#     starting_point=None
# )
# print(results_2)

results_1 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

100%|██████████| 98/98 [00:00<00:00, 334.71it/s]


{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.6888889074325562, 'correct_tokens': 62, 'tot_tokens': 90}}


100%|██████████| 1264/1264 [00:12<00:00, 101.19it/s]

{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.002572898752987385, 'correct_tokens': 3, 'tot_tokens': 1166}}


In [8]:
# a = a + 1

In [9]:
model.save_pretrained('memo_example')
tokenizer.save_pretrained('memo_example')

model2 = MeMoForCausalLM.from_pretrained('memo_example') #device_map='auto')
model2.to('cuda')
model2.train()
tokenizer2 = MeMoTokenizer.from_pretrained('memo_example')
model2

MeMo embedding initilialization
MeMo embedding initilialization
Setting pad token and pad token id = <|endoftext|>, 0


MeMoForCausalLM(
  (memo): MeMo(
    (encoder): MeMoEmbedding(50277, 2048, padding_idx=0)
    (layers): MeMoLayers(
      (0): MeMoLayer(
        (W_v_single_head): ProjectionTokens(in_features=2048, out_features=512)
        (Prj): ProjectionSequence((trasposed wrt saved one) in_features=8192, out_features=2048)
        (CMM): CorrelationMatrixMemory(in_features=2048, out_features=2048)
      )
    )
  )
  (lm_head): MeMoEmbedding(50277, 2048, padding_idx=0)
)

In [10]:
print(model.memo.device)
print(model2.memo.device)

cuda:0
cuda:0


In [11]:
# results = perform_evaluation(
#     model=model2,
#     tokenizer=tokenizer,
#     text=first_text*8
# )
# print(results)

results_1 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

100%|██████████| 98/98 [00:00<00:00, 314.09it/s]


{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.6888889074325562, 'correct_tokens': 62, 'tot_tokens': 90}}


100%|██████████| 1264/1264 [00:12<00:00, 102.34it/s]

{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.002572898752987385, 'correct_tokens': 3, 'tot_tokens': 1166}}


In [12]:
# results = perform_evaluation(
#     model=model2,
#     tokenizer=tokenizer2,
#     text=first_text*1
# )
# print(results)

In [13]:
model2.memorize_text(memo_input_2)

results_1 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

100%|██████████| 98/98 [00:00<00:00, 323.30it/s]


{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.6777777671813965, 'correct_tokens': 61, 'tot_tokens': 90}}


100%|██████████| 1264/1264 [00:12<00:00, 102.13it/s]

{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.6620926260948181, 'correct_tokens': 772, 'tot_tokens': 1166}}


In [14]:
model2.forget_text(memo_input_2)

results_1 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_first_text]*1
)
print(results_1)

results_2 = perform_evaluation(
    model=model2,
    tokenizer=tokenizer,
    text=[my_second_text]*1
)
print(results_2)

100%|██████████| 98/98 [00:00<00:00, 332.35it/s]


{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.6888889074325562, 'correct_tokens': 62, 'tot_tokens': 90}}


100%|██████████| 1264/1264 [00:12<00:00, 101.36it/s]

{'out_loss': tensor(nan, device='cuda:0'), 'token_accuracy': {'accuracy': 0.002572898752987385, 'correct_tokens': 3, 'tot_tokens': 1166}}


In [15]:
# print(torch.exp(results['out_loss']))
# print(results)

In [16]:
tokenizer.pad_token_type_id

0

In [ ]:
exit()

: 